In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.datasets as datasets
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, random_split
import time
import matplotlib.pyplot as plt
import os
from sklearn.model_selection import train_test_split
from torch.utils.data import Subset
import torch

import pickle

In [2]:
import pyro
import pyro.distributions as dist
from pyro.nn import PyroModule, PyroSample

c:\Users\Revalda Putawara\.conda\envs\bnntest\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import pandas as pd

In [4]:
device = torch.device("cuda")

In [5]:
from torchvision.datasets import ImageFolder

In [6]:
dataset = ImageFolder(
    root="data/shipsnet/foldered",
    transform=transforms.ToTensor()
)

In [7]:
loader = DataLoader(
    dataset,
    batch_size=64,
    shuffle=False,
    num_workers=1)

In [8]:
shipsnet_mean = [0.4119, 0.4243, 0.3724]
shipsnet_std = [0.1899, 0.1569, 0.1515]

def load_data(batch_size=16):
    transform = transforms.Compose([
        transforms.Resize((64, 64)),
        transforms.ToTensor(),
        transforms.Normalize(mean=shipsnet_mean, 
                             std=shipsnet_std)
    ])

    dataset = ImageFolder(
    root="data/shipsnet/foldered",
    transform=transform
    )

    torch.manual_seed(42)

    #train_size = int(0.8 * len(dataset))
    #test_size = len(dataset) - train_size
    #train_dataset, test_dataset = random_split(dataset, [train_size, test_size])
    
    with open('datasplit/shipsnet_split_indices.pkl', 'rb') as f:
        split = pickle.load(f)
        train_dataset = Subset(dataset, split['train'])
        test_dataset = Subset(dataset, split['test'])

    # Add num_workers and pin_memory for faster data loading
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, 
                             num_workers=4, pin_memory=True, persistent_workers=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size,
                            num_workers=4, pin_memory=True, persistent_workers=True)
    return train_loader, test_loader, train_dataset, test_dataset

In [9]:
train_loader, test_loader, train_ds, test_ds = load_data(16)

In [10]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import transforms

In [11]:
lr          = 1e-3
num_epochs  = 20

In [12]:
# ─── 5. Model Definition ──────────────────────────────────────────────────────
import torch
import torch.nn as nn
import torch.nn.functional as F

class ShipsCNN(nn.Module):
    def __init__(self, num_classes=2, activation='relu'):
        super().__init__()

        # Activation setup (same as BayesShipsCNN)
        act_map = {
            'relu': F.relu,
            'tanh': torch.tanh,
            'sigmoid': torch.sigmoid,
            'sin': torch.sin,
            'relu6': F.relu6,
            'leaky_relu': F.leaky_relu,
            'selu': F.selu,
        }
        if activation not in act_map:
            raise ValueError(f"Unsupported activation: {activation}")
        self.activation_fn = act_map[activation]

        # Layers: Same as BayesShipsCNN (2 conv layers + pooling)
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)

        # Fully connected layer
        # BayesShipsCNN flattens [B,64,16,16] → fc1: 64*16*16 → 2
        # Assuming input image size = 64x64 (same assumption as BayesShipsCNN)
        self.fc1 = nn.Linear(64 * 16 * 16, num_classes)

    def forward(self, x):
        x = self.activation_fn(self.conv1(x))
        x = self.pool(x)
        x = self.activation_fn(self.conv2(x))
        x = self.pool(x)
        x = x.view(x.size(0), -1)
        logits = self.fc1(x)
        return logits


# Example usage
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ShipsCNN(num_classes=2, activation='relu').to(device)
print(model)


# ─── 6. Loss, Optimizer & Scheduler ───────────────────────────────────────────
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=lr)
#scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.1)

# ─── 7. Training Loop ─────────────────────────────────────────────────────────
for epoch in range(1, num_epochs + 1):
    # — Train —
    model.train()
    running_loss = running_corrects = 0
    for imgs, labels in train_loader:
        imgs  = imgs.to(device, non_blocking=True)
        labels= labels.to(device, non_blocking=True)

        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        preds = outputs.argmax(dim=1)
        running_loss    += loss.item() * imgs.size(0)
        running_corrects+= (preds == labels).sum().item()

    epoch_loss = running_loss / len(train_ds)
    epoch_acc  = running_corrects / len(train_ds)

    # — Validate —
    model.eval()
    val_loss = val_corrects = 0
    with torch.no_grad():
        for imgs, labels in test_loader:
            imgs   = imgs.to(device)
            labels = labels.to(device)
            outputs= model(imgs)
            loss   = criterion(outputs, labels)
            preds  = outputs.argmax(dim=1)

            val_loss     += loss.item() * imgs.size(0)
            val_corrects += (preds == labels).sum().item()

    val_loss = val_loss / len(test_ds)
    val_acc  = val_corrects / len(test_ds)
    #scheduler.step()

    print(f"Epoch {epoch:2d}/{num_epochs} "
          f"Train: loss={epoch_loss:.4f}, acc={epoch_acc:.4f} | "
          f"Val:   loss={val_loss:.4f}, acc={val_acc:.4f}")

# ─── 8. Save Checkpoint ───────────────────────────────────────────────────────
#os.makedirs("checkpoints", exist_ok=True)
#torch.save(model.state_dict(), "checkpoints/shipsnet_cnn.pth")
#print("Model saved to checkpoints/shipsnet_cnn.pth")

ShipsCNN(
  (conv1): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (fc1): Linear(in_features=16384, out_features=2, bias=True)
)
Epoch  1/20 Train: loss=0.2229, acc=0.9169 | Val:   loss=0.1467, acc=0.9363
Epoch  2/20 Train: loss=0.1103, acc=0.9653 | Val:   loss=0.1165, acc=0.9587
Epoch  3/20 Train: loss=0.0682, acc=0.9788 | Val:   loss=0.0813, acc=0.9675
Epoch  4/20 Train: loss=0.0561, acc=0.9806 | Val:   loss=0.1078, acc=0.9688
Epoch  5/20 Train: loss=0.0372, acc=0.9869 | Val:   loss=0.0796, acc=0.9725
Epoch  6/20 Train: loss=0.0213, acc=0.9931 | Val:   loss=0.0717, acc=0.9838
Epoch  7/20 Train: loss=0.0183, acc=0.9931 | Val:   loss=0.1027, acc=0.9750
Epoch  8/20 Train: loss=0.0094, acc=0.9966 | Val:   loss=0.0990, acc=0.9725
Epoch  9/20 Train: loss=0.0297, acc=0.9897 | Val:   loss=0.1975, acc=0.9487
Epoc

In [13]:
# test the model on the test set
model.eval()
test_loss = test_corrects = 0
with torch.no_grad():
    for imgs, labels in test_loader:
        imgs   = imgs.to(device)
        labels = labels.to(device)
        outputs= model(imgs)
        loss   = criterion(outputs, labels)
        preds  = outputs.argmax(dim=1)

        test_loss     += loss.item() * imgs.size(0)
        test_corrects += (preds == labels).sum().item()
test_loss = test_loss / len(test_ds)
test_acc  = test_corrects / len(test_ds)

print(f"Test: loss={test_loss:.4f}, acc={test_acc:.4f}")

Test: loss=0.1179, acc=0.9788


In [ ]:
# ─── 6. Save Model ─────────────────────────────────────────────────────────────
#os.makedirs("checkpoints", exist_ok=True)
#torch.save(model.state_dict(), "checkpoints/shipsnet_cnn.pth")
#print("Training complete. Model saved to checkpoints/shipsnet_cnn.pth")

In [14]:
#print total trainable parameters
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total trainable parameters: {total_params:,}")

Total trainable parameters: 52,162


## NEW MODEL